# RavenStack SaaS Analytics & Engineering Workflow
This Jupyter Notebook consolidates the complete database setup, data cleaning, exploratory data analysis (EDA), and data export workflow in sequential order:
1. **Part 1:** Schema Definition & Data Ingestion (`Tables & Importing.txt`)
2. **Part 2:** Exploratory Data Analysis & Business Queries (`ravenstack_eda_queries.sql`)
3. **Part 3:** Data Cleaning, Validation & Export (`saas_project.sql`)

---
## 1. Environment & Setup (Optional `%load_ext sql`)
If executing directly in Jupyter with a MySQL kernel/extension, you can enable SQL cell magics:

In [ ]:
# Uncomment and run if running inside Jupyter with ipython-sql and MySQL connection:
# !pip install ipython-sql pymysql cryptography
# %load_ext sql
# %sql mysql+pymysql://<user>:<password>@localhost:3306/ravenstack_db

---
# PART 1: Tables & Importing
*(Source: `Tables & Importing.txt`)*

Database initialization, table schema creation with primary/foreign keys, and data loading via `LOAD DATA INFILE`.

### 1.1 Database Initialization & Teardown

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS ravenstack_db;
USE ravenstack_db;

-- Drop tables in reverse dependency order to avoid foreign key errors on re-run
DROP TABLE IF EXISTS feature_usage;
DROP TABLE IF EXISTS support_tickets;
DROP TABLE IF EXISTS churn_events;
DROP TABLE IF EXISTS subscriptions;
DROP TABLE IF EXISTS accounts;

### 1.2 Table Definitions (DDL)
Creating tables: `accounts`, `subscriptions`, `churn_events`, `feature_usage`, and `support_tickets`.

In [ ]:
%%sql
-- ===================================================
-- 1. ACCOUNTS
-- ===================================================
CREATE TABLE accounts (
    account_id VARCHAR(10) PRIMARY KEY,
    account_name VARCHAR(50),
    industry VARCHAR(30),
    country VARCHAR(5),
    signup_date DATE,
    referral_source VARCHAR(20),
    plan_tier VARCHAR(20),
    seats INT,
    is_trial BOOLEAN,
    churn_flag BOOLEAN
);

-- ===================================================
-- 2. SUBSCRIPTIONS
-- ===================================================
CREATE TABLE subscriptions (
    subscription_id VARCHAR(10) PRIMARY KEY,
    account_id VARCHAR(10),
    start_date DATE,
    end_date DATE,
    plan_tier VARCHAR(20),
    seats INT,
    mrr_amount INT,
    arr_amount INT,
    is_trial BOOLEAN,
    upgrade_flag BOOLEAN,
    downgrade_flag BOOLEAN,
    churn_flag BOOLEAN,
    billing_frequency VARCHAR(15),
    auto_renew_flag BOOLEAN,
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
);

-- ===================================================
-- 3. CHURN EVENTS
-- ===================================================
CREATE TABLE churn_events (
    churn_event_id VARCHAR(10) PRIMARY KEY,
    account_id VARCHAR(10),
    churn_date DATE,
    reason_code VARCHAR(30),
    refund_amount_usd DECIMAL(10, 2),
    preceding_upgrade_flag BOOLEAN,
    preceding_downgrade_flag BOOLEAN,
    is_reactivation BOOLEAN,
    feedback_text TEXT,
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
);

-- ===================================================
-- 4. FEATURE USAGE
-- ===================================================
CREATE TABLE feature_usage (
    usage_id VARCHAR(10),
    subscription_id VARCHAR(10),
    usage_date DATE,
    feature_name VARCHAR(30),
    usage_count INT,
    usage_duration_secs INT,
    error_count INT,
    is_beta_feature BOOLEAN,
    FOREIGN KEY (subscription_id) REFERENCES subscriptions(subscription_id)
);

-- ===================================================
-- 5. SUPPORT TICKETS
-- ===================================================
CREATE TABLE support_tickets (
    ticket_id VARCHAR(10) PRIMARY KEY,
    account_id VARCHAR(10),
    submitted_at DATETIME,
    closed_at DATETIME,
    resolution_time_hours DECIMAL(10, 2),
    priority VARCHAR(15),
    first_response_time_minutes INT,
    satisfaction_score DECIMAL(3, 1),
    escalation_flag BOOLEAN,
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
);

### 1.3 Data Loading (`LOAD DATA INFILE`)

In [ ]:
%%sql
SET FOREIGN_KEY_CHECKS = 0;

-- ---------------------------------------------------
-- 1. LOAD ravenstack_accounts.csv
-- ---------------------------------------------------
LOAD DATA INFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/ravenstack_accounts.csv'
INTO TABLE accounts
FIELDS TERMINATED BY ',' 
ENCLOSED BY '"' 
LINES TERMINATED BY '\n'
IGNORE 1 ROWS
(
    @account_id, @account_name, @industry, @country, @signup_date, 
    @referral_source, @plan_tier, @seats, @is_trial, @churn_flag
)
SET 
    account_id      = NULLIF(NULLIF(TRIM(@account_id), ''), '\\N'),
    account_name    = NULLIF(NULLIF(TRIM(@account_name), ''), '\\N'),
    industry        = NULLIF(NULLIF(TRIM(@industry), ''), '\\N'),
    country         = NULLIF(NULLIF(TRIM(@country), ''), '\\N'),
    signup_date     = NULLIF(NULLIF(TRIM(@signup_date), ''), '\\N'),
    referral_source = NULLIF(NULLIF(TRIM(@referral_source), ''), '\\N'),
    plan_tier       = NULLIF(NULLIF(TRIM(@plan_tier), ''), '\\N'),
    seats           = NULLIF(NULLIF(TRIM(@seats), ''), '\\N'),
    is_trial        = CASE WHEN TRIM(@is_trial) = 'True' THEN 1 ELSE 0 END,
    churn_flag      = CASE WHEN TRIM(@churn_flag) = 'True' THEN 1 ELSE 0 END;

-- ---------------------------------------------------
-- 2. LOAD ravenstack_subscriptions.csv
-- ---------------------------------------------------
LOAD DATA INFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/ravenstack_subscriptions.csv'
INTO TABLE subscriptions
FIELDS TERMINATED BY ',' 
ENCLOSED BY '"' 
LINES TERMINATED BY '\n'
IGNORE 1 ROWS
(
    @subscription_id, @account_id, @start_date, @end_date, @plan_tier, 
    @seats, @mrr_amount, @arr_amount, @is_trial, @upgrade_flag, 
    @downgrade_flag, @churn_flag, @billing_frequency, @auto_renew_flag
)
SET 
    subscription_id   = NULLIF(NULLIF(TRIM(@subscription_id), ''), '\\N'),
    account_id        = NULLIF(NULLIF(TRIM(@account_id), ''), '\\N'),
    start_date        = NULLIF(NULLIF(TRIM(@start_date), ''), '\\N'),
    end_date          = NULLIF(NULLIF(TRIM(@end_date), ''), '\\N'),
    plan_tier         = NULLIF(NULLIF(TRIM(@plan_tier), ''), '\\N'),
    seats             = NULLIF(NULLIF(TRIM(@seats), ''), '\\N'),
    mrr_amount        = NULLIF(NULLIF(TRIM(@mrr_amount), ''), '\\N'),
    arr_amount        = NULLIF(NULLIF(TRIM(@arr_amount), ''), '\\N'),
    is_trial          = CASE WHEN TRIM(@is_trial) = 'True' THEN 1 ELSE 0 END,
    upgrade_flag      = CASE WHEN TRIM(@upgrade_flag) = 'True' THEN 1 ELSE 0 END,
    downgrade_flag    = CASE WHEN TRIM(@downgrade_flag) = 'True' THEN 1 ELSE 0 END,
    churn_flag        = CASE WHEN TRIM(@churn_flag) = 'True' THEN 1 ELSE 0 END,
    billing_frequency = NULLIF(NULLIF(TRIM(@billing_frequency), ''), '\\N'),
    auto_renew_flag   = CASE WHEN TRIM(@auto_renew_flag) = 'True' THEN 1 ELSE 0 END;

-- ---------------------------------------------------
-- 3. LOAD ravenstack_churn_events.csv
-- ---------------------------------------------------
LOAD DATA INFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/ravenstack_churn_events.csv'
INTO TABLE churn_events
FIELDS TERMINATED BY ',' 
ENCLOSED BY '"' 
LINES TERMINATED BY '\n'
IGNORE 1 ROWS
(
    @churn_event_id, @account_id, @churn_date, @reason_code, @refund_amount_usd, 
    @preceding_upgrade_flag, @preceding_downgrade_flag, @is_reactivation, @feedback_text
)
SET 
    churn_event_id           = NULLIF(NULLIF(TRIM(@churn_event_id), ''), '\\N'),
    account_id               = NULLIF(NULLIF(TRIM(@account_id), ''), '\\N'),
    churn_date               = NULLIF(NULLIF(TRIM(@churn_date), ''), '\\N'),
    reason_code              = NULLIF(NULLIF(TRIM(@reason_code), ''), '\\N'),
    refund_amount_usd        = NULLIF(NULLIF(TRIM(@refund_amount_usd), ''), '\\N'),
    preceding_upgrade_flag   = CASE WHEN TRIM(@preceding_upgrade_flag) = 'True' THEN 1 ELSE 0 END,
    preceding_downgrade_flag = CASE WHEN TRIM(@preceding_downgrade_flag) = 'True' THEN 1 ELSE 0 END,
    is_reactivation          = CASE WHEN TRIM(@is_reactivation) = 'True' THEN 1 ELSE 0 END,
    feedback_text            = NULLIF(NULLIF(TRIM(@feedback_text), ''), '\\N');

-- ---------------------------------------------------
-- 4. LOAD ravenstack_feature_usage.csv
-- ---------------------------------------------------
LOAD DATA INFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/ravenstack_feature_usage.csv'
INTO TABLE feature_usage
FIELDS TERMINATED BY ',' 
ENCLOSED BY '"' 
LINES TERMINATED BY '\n'
IGNORE 1 ROWS
(
    @usage_id, @subscription_id, @usage_date, @feature_name, 
    @usage_count, @usage_duration_secs, @error_count, @is_beta_feature
)
SET 
    usage_id            = NULLIF(NULLIF(TRIM(@usage_id), ''), '\\N'),
    subscription_id     = NULLIF(NULLIF(TRIM(@subscription_id), ''), '\\N'),
    usage_date          = NULLIF(NULLIF(TRIM(@usage_date), ''), '\\N'),
    feature_name        = NULLIF(NULLIF(TRIM(@feature_name), ''), '\\N'),
    usage_count         = NULLIF(NULLIF(TRIM(@usage_count), ''), '\\N'),
    usage_duration_secs = NULLIF(NULLIF(TRIM(@usage_duration_secs), ''), '\\N'),
    error_count         = NULLIF(NULLIF(TRIM(@error_count), ''), '\\N'),
    is_beta_feature     = CASE WHEN TRIM(@is_beta_feature) = 'True' THEN 1 ELSE 0 END;

-- ---------------------------------------------------
-- 5. LOAD ravenstack_support_tickets.csv
-- ---------------------------------------------------
LOAD DATA INFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/ravenstack_support_tickets.csv'
INTO TABLE support_tickets
FIELDS TERMINATED BY ',' 
ENCLOSED BY '"' 
LINES TERMINATED BY '\n'
IGNORE 1 ROWS
(
    @ticket_id, @account_id, @submitted_at, @closed_at, @resolution_time_hours, 
    @priority, @first_response_time_minutes, @satisfaction_score, @escalation_flag
)
SET 
    ticket_id                   = NULLIF(NULLIF(TRIM(@ticket_id), ''), '\\N'),
    account_id                  = NULLIF(NULLIF(TRIM(@account_id), ''), '\\N'),
    submitted_at                = NULLIF(NULLIF(TRIM(@submitted_at), ''), '\\N'),
    closed_at                   = NULLIF(NULLIF(TRIM(@closed_at), ''), '\\N'),
    resolution_time_hours       = NULLIF(NULLIF(TRIM(@resolution_time_hours), ''), '\\N'),
    priority                    = NULLIF(NULLIF(TRIM(@priority), ''), '\\N'),
    first_response_time_minutes = NULLIF(NULLIF(TRIM(@first_response_time_minutes), ''), '\\N'),
    satisfaction_score          = NULLIF(NULLIF(TRIM(@satisfaction_score), ''), '\\N'),
    escalation_flag             = CASE WHEN TRIM(@escalation_flag) = 'True' THEN 1 ELSE 0 END;

SET FOREIGN_KEY_CHECKS = 1;

### 1.4 Post-Load Verification Row Counts

In [ ]:
%%sql
SELECT 'accounts' AS TableName, COUNT(*) AS TotalRows FROM accounts
UNION ALL
SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALL
SELECT 'churn_events', COUNT(*) FROM churn_events
UNION ALL
SELECT 'feature_usage', COUNT(*) FROM feature_usage
UNION ALL
SELECT 'support_tickets', COUNT(*) FROM support_tickets;

---
# PART 2: Exploratory Data Analysis (EDA)
*(Source: `ravenstack_eda_queries.sql`)*

Comprehensive exploration of data distribution, integrity, customer subscriptions, feature adoption, support load, and churn.

### 2.1 Section 1: Basic Data Understanding & Schema Inspection

In [ ]:
%%sql
USE ravenstack_db;

-- Step 1.1 — Show all tables
SHOW TABLES;

-- Step 1.2 — Row counts in every table
SELECT 'accounts' AS table_name, COUNT(*) AS total_rows FROM accounts
UNION ALL SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALL SELECT 'churn_events', COUNT(*) FROM churn_events
UNION ALL SELECT 'feature_usage', COUNT(*) FROM feature_usage
UNION ALL SELECT 'support_tickets', COUNT(*) FROM support_tickets;

-- Step 1.3 — Sample records from each table
SELECT * FROM accounts LIMIT 5;
SELECT * FROM subscriptions LIMIT 5;
SELECT * FROM churn_events LIMIT 5;
SELECT * FROM feature_usage LIMIT 5;
SELECT * FROM support_tickets LIMIT 5;

### 2.2 Table Structures, Primary Keys & Orphan Foreign Keys

In [ ]:
%%sql
-- Step 1.4 — Columns of each table (structure)
DESCRIBE accounts;
DESCRIBE subscriptions;
DESCRIBE churn_events;
DESCRIBE feature_usage;
DESCRIBE support_tickets;

-- Step 1.5 — Primary keys
SHOW KEYS FROM accounts WHERE Key_name = 'PRIMARY';
SHOW KEYS FROM subscriptions WHERE Key_name = 'PRIMARY';
SHOW KEYS FROM churn_events WHERE Key_name = 'PRIMARY';
SHOW KEYS FROM feature_usage WHERE Key_name = 'PRIMARY';
SHOW KEYS FROM support_tickets WHERE Key_name = 'PRIMARY';

-- Step 1.6 — Unique ID counts
SELECT COUNT(DISTINCT account_id) AS unique_account_ids FROM accounts;
SELECT COUNT(DISTINCT subscription_id) AS unique_subscription_ids FROM subscriptions;
SELECT COUNT(DISTINCT churn_event_id) AS unique_churn_event_ids FROM churn_events;
SELECT COUNT(DISTINCT usage_id) AS unique_usage_ids FROM feature_usage;
SELECT COUNT(DISTINCT ticket_id) AS unique_ticket_ids FROM support_tickets;

-- Step 1.7 — Foreign key checks: rows pointing to a parent that doesn't exist
SELECT s.subscription_id, s.account_id
FROM subscriptions s LEFT JOIN accounts a ON s.account_id = a.account_id
WHERE a.account_id IS NULL;

SELECT c.churn_event_id, c.account_id
FROM churn_events c LEFT JOIN accounts a ON c.account_id = a.account_id
WHERE a.account_id IS NULL;

SELECT t.ticket_id, t.account_id
FROM support_tickets t LEFT JOIN accounts a ON t.account_id = a.account_id
WHERE a.account_id IS NULL;

SELECT f.usage_id, f.subscription_id
FROM feature_usage f LEFT JOIN subscriptions s ON f.subscription_id = s.subscription_id
WHERE s.subscription_id IS NULL;

### 2.3 Section 2: Data Quality Checks (Duplicates, Nulls, Ranges & Outliers)

In [ ]:
%%sql
-- Step 2.1 — Duplicate primary keys
SELECT account_id, COUNT(*) AS cnt FROM accounts GROUP BY account_id HAVING COUNT(*) > 1;
SELECT subscription_id, COUNT(*) AS cnt FROM subscriptions GROUP BY subscription_id HAVING COUNT(*) > 1;
SELECT churn_event_id, COUNT(*) AS cnt FROM churn_events GROUP BY churn_event_id HAVING COUNT(*) > 1;
SELECT usage_id, COUNT(*) AS cnt FROM feature_usage GROUP BY usage_id HAVING COUNT(*) > 1;
SELECT ticket_id, COUNT(*) AS cnt FROM support_tickets GROUP BY ticket_id HAVING COUNT(*) > 1;

-- Step 2.2 — NULL / blank checks in key columns
SELECT
  SUM(account_name IS NULL) AS null_account_name,
  SUM(industry IS NULL OR industry = '') AS missing_industry,
  SUM(country IS NULL OR country = '') AS missing_country,
  SUM(signup_date IS NULL) AS null_signup_date,
  SUM(plan_tier IS NULL) AS null_plan_tier,
  SUM(seats IS NULL) AS null_seats
FROM accounts;

SELECT
  SUM(mrr_amount IS NULL) AS null_mrr,
  SUM(arr_amount IS NULL) AS null_arr,
  SUM(end_date IS NULL) AS null_end_date,
  SUM(billing_frequency IS NULL) AS null_billing_frequency
FROM subscriptions;

SELECT
  SUM(feedback_text IS NULL) AS null_feedback,
  SUM(refund_amount_usd IS NULL) AS null_refund,
  SUM(reason_code IS NULL) AS null_reason_code
FROM churn_events;

SELECT
  SUM(usage_count IS NULL) AS null_usage_count,
  SUM(error_count IS NULL) AS null_error_count
FROM feature_usage;

SELECT
  SUM(satisfaction_score IS NULL) AS null_satisfaction,
  SUM(resolution_time_hours IS NULL) AS null_resolution,
  SUM(closed_at IS NULL) AS null_closed_at
FROM support_tickets;

-- Step 2.3 — Distinct values in categorical columns
SELECT industry, COUNT(*) AS cnt FROM accounts GROUP BY industry ORDER BY cnt DESC;
SELECT country, COUNT(*) AS cnt FROM accounts GROUP BY country ORDER BY cnt DESC;
SELECT plan_tier, COUNT(*) AS cnt FROM accounts GROUP BY plan_tier ORDER BY cnt DESC;
SELECT referral_source, COUNT(*) AS cnt FROM accounts GROUP BY referral_source ORDER BY cnt DESC;
SELECT billing_frequency, COUNT(*) AS cnt FROM subscriptions GROUP BY billing_frequency ORDER BY cnt DESC;
SELECT reason_code, COUNT(*) AS cnt FROM churn_events GROUP BY reason_code ORDER BY cnt DESC;
SELECT feature_name, COUNT(*) AS cnt FROM feature_usage GROUP BY feature_name ORDER BY cnt DESC;
SELECT priority, COUNT(*) AS cnt FROM support_tickets GROUP BY priority ORDER BY cnt DESC;

-- Step 2.4 — Min/Max of important numeric columns
SELECT MIN(seats) AS min_seats, MAX(seats) AS max_seats, AVG(seats) AS avg_seats FROM accounts;
SELECT MIN(mrr_amount) AS min_mrr, MAX(mrr_amount) AS max_mrr,
       MIN(arr_amount) AS min_arr, MAX(arr_amount) AS max_arr
FROM subscriptions;
SELECT MIN(refund_amount_usd) AS min_refund, MAX(refund_amount_usd) AS max_refund FROM churn_events;
SELECT MIN(usage_count) AS min_usage, MAX(usage_count) AS max_usage,
       MIN(error_count) AS min_errors, MAX(error_count) AS max_errors
FROM feature_usage;
SELECT MIN(resolution_time_hours) AS min_res, MAX(resolution_time_hours) AS max_res,
       MIN(satisfaction_score) AS min_sat, MAX(satisfaction_score) AS max_sat
FROM support_tickets;

-- Step 2.5 — Suspicious values
SELECT COUNT(*) AS zero_or_negative_seats FROM accounts WHERE seats <= 0;
SELECT COUNT(*) AS negative_mrr FROM subscriptions WHERE mrr_amount < 0;
SELECT COUNT(*) AS negative_refund FROM churn_events WHERE refund_amount_usd < 0;
SELECT COUNT(*) AS negative_satisfaction FROM support_tickets WHERE satisfaction_score < 0;

-- Step 2.6 & 2.7 — Date validity and time boundaries
SELECT COUNT(*) AS bad_subscription_dates FROM subscriptions WHERE end_date < start_date;
SELECT COUNT(*) AS bad_ticket_dates FROM support_tickets WHERE closed_at < submitted_at;
SELECT MIN(signup_date) AS earliest_signup, MAX(signup_date) AS latest_signup FROM accounts;
SELECT MIN(start_date) AS earliest_sub, MAX(start_date) AS latest_sub FROM subscriptions;
SELECT MIN(churn_date) AS earliest_churn, MAX(churn_date) AS latest_churn FROM churn_events;
SELECT MIN(usage_date) AS earliest_usage, MAX(usage_date) AS latest_usage FROM feature_usage;
SELECT MIN(submitted_at) AS earliest_ticket, MAX(submitted_at) AS latest_ticket FROM support_tickets;

### 2.4 Sections 3–7: Domain-Specific EDA (Accounts, Subscriptions, Churn, Usage, Tickets)

In [ ]:
%%sql
-- ============================================================
-- SECTION 3: ACCOUNTS ANALYSIS
-- ============================================================
SELECT COUNT(*) AS total_accounts FROM accounts;
SELECT industry, COUNT(*) AS num_accounts FROM accounts GROUP BY industry ORDER BY num_accounts DESC;
SELECT country, COUNT(*) AS num_accounts FROM accounts GROUP BY country ORDER BY num_accounts DESC;
SELECT plan_tier, COUNT(*) AS num_accounts FROM accounts GROUP BY plan_tier ORDER BY num_accounts DESC;
SELECT referral_source, COUNT(*) AS num_accounts FROM accounts GROUP BY referral_source ORDER BY num_accounts DESC;
SELECT is_trial, COUNT(*) AS num_accounts FROM accounts GROUP BY is_trial;
SELECT AVG(seats) AS avg_seats, MIN(seats) AS min_seats, MAX(seats) AS max_seats FROM accounts;
SELECT churn_flag, COUNT(*) AS num_accounts FROM accounts GROUP BY churn_flag;

-- Signup trend by year/month
SELECT YEAR(signup_date) AS signup_year, MONTH(signup_date) AS signup_month, COUNT(*) AS num_signups
FROM accounts
GROUP BY YEAR(signup_date), MONTH(signup_date)
ORDER BY signup_year, signup_month;

-- ============================================================
-- SECTION 4: SUBSCRIPTION ANALYSIS
-- ============================================================
SELECT COUNT(*) AS total_subscriptions FROM subscriptions;
SELECT plan_tier, COUNT(*) AS num_subs FROM subscriptions GROUP BY plan_tier ORDER BY num_subs DESC;
SELECT is_trial, COUNT(*) AS num_subs FROM subscriptions GROUP BY is_trial;
SELECT billing_frequency, COUNT(*) AS num_subs FROM subscriptions GROUP BY billing_frequency;
SELECT auto_renew_flag, COUNT(*) AS num_subs FROM subscriptions GROUP BY auto_renew_flag;
SELECT upgrade_flag, COUNT(*) AS num_subs FROM subscriptions GROUP BY upgrade_flag;
SELECT downgrade_flag, COUNT(*) AS num_subs FROM subscriptions GROUP BY downgrade_flag;
SELECT churn_flag, COUNT(*) AS num_subs FROM subscriptions GROUP BY churn_flag;
SELECT AVG(mrr_amount) AS avg_mrr, SUM(mrr_amount) AS total_mrr FROM subscriptions;
SELECT AVG(arr_amount) AS avg_arr, SUM(arr_amount) AS total_arr FROM subscriptions;
SELECT AVG(seats) AS avg_seats FROM subscriptions;

-- Subscription trend over time
SELECT YEAR(start_date) AS sub_year, MONTH(start_date) AS sub_month, COUNT(*) AS num_new_subs
FROM subscriptions
GROUP BY YEAR(start_date), MONTH(start_date)
ORDER BY sub_year, sub_month;

-- Revenue by plan & upgrades/downgrades
SELECT plan_tier, SUM(mrr_amount) AS total_mrr, SUM(arr_amount) AS total_arr FROM subscriptions GROUP BY plan_tier ORDER BY total_mrr DESC;
SELECT plan_tier, SUM(upgrade_flag) AS total_upgrades FROM subscriptions GROUP BY plan_tier ORDER BY total_upgrades DESC;
SELECT plan_tier, SUM(downgrade_flag) AS total_downgrades FROM subscriptions GROUP BY plan_tier ORDER BY total_downgrades DESC;

-- ============================================================
-- SECTION 5: CHURN ANALYSIS
-- ============================================================
SELECT COUNT(*) AS total_churn_events FROM churn_events;
SELECT reason_code, COUNT(*) AS num_churns FROM churn_events GROUP BY reason_code ORDER BY num_churns DESC;
SELECT a.plan_tier, COUNT(*) AS num_churns FROM churn_events c JOIN accounts a ON c.account_id = a.account_id GROUP BY a.plan_tier ORDER BY num_churns DESC;
SELECT a.industry, COUNT(*) AS num_churns FROM churn_events c JOIN accounts a ON c.account_id = a.account_id GROUP BY a.industry ORDER BY num_churns DESC;
SELECT a.country, COUNT(*) AS num_churns FROM churn_events c JOIN accounts a ON c.account_id = a.account_id GROUP BY a.country ORDER BY num_churns DESC;
SELECT YEAR(churn_date) AS churn_year, MONTH(churn_date) AS churn_month, COUNT(*) AS num_churns FROM churn_events GROUP BY YEAR(churn_date), MONTH(churn_date) ORDER BY churn_year, churn_month;
SELECT is_reactivation, COUNT(*) AS num_events FROM churn_events GROUP BY is_reactivation;
SELECT preceding_upgrade_flag, COUNT(*) AS num_events FROM churn_events GROUP BY preceding_upgrade_flag;
SELECT preceding_downgrade_flag, COUNT(*) AS num_events FROM churn_events GROUP BY preceding_downgrade_flag;
SELECT SUM(refund_amount_usd) AS total_refunds, AVG(refund_amount_usd) AS avg_refund FROM churn_events;

-- ============================================================
-- SECTION 6: FEATURE USAGE ANALYSIS
-- ============================================================
SELECT COUNT(*) AS total_usage_records FROM feature_usage;
SELECT feature_name, SUM(usage_count) AS total_usage FROM feature_usage GROUP BY feature_name ORDER BY total_usage DESC;
SELECT feature_name, AVG(usage_count) AS avg_usage FROM feature_usage GROUP BY feature_name ORDER BY avg_usage DESC;
SELECT feature_name, SUM(usage_duration_secs) AS total_duration FROM feature_usage GROUP BY feature_name ORDER BY total_duration DESC;
SELECT feature_name, SUM(error_count) AS total_errors FROM feature_usage GROUP BY feature_name ORDER BY total_errors DESC;
SELECT is_beta_feature, COUNT(*) AS num_records FROM feature_usage GROUP BY is_beta_feature;
SELECT YEAR(usage_date) AS usage_year, MONTH(usage_date) AS usage_month, COUNT(*) AS num_records FROM feature_usage GROUP BY YEAR(usage_date), MONTH(usage_date) ORDER BY usage_year, usage_month;

-- ============================================================
-- SECTION 7: SUPPORT TICKET ANALYSIS
-- ============================================================
SELECT COUNT(*) AS total_tickets FROM support_tickets;
SELECT priority, COUNT(*) AS num_tickets FROM support_tickets GROUP BY priority ORDER BY num_tickets DESC;
SELECT account_id, COUNT(*) AS num_tickets FROM support_tickets GROUP BY account_id ORDER BY num_tickets DESC LIMIT 10;
SELECT AVG(resolution_time_hours) AS avg_resolution, MIN(resolution_time_hours) AS min_resolution, MAX(resolution_time_hours) AS max_resolution FROM support_tickets;
SELECT AVG(first_response_time_minutes) AS avg_first_response FROM support_tickets;
SELECT AVG(satisfaction_score) AS avg_satisfaction FROM support_tickets;
SELECT escalation_flag, COUNT(*) AS num_tickets FROM support_tickets GROUP BY escalation_flag;
SELECT YEAR(submitted_at) AS ticket_year, MONTH(submitted_at) AS ticket_month, COUNT(*) AS num_tickets FROM support_tickets GROUP BY YEAR(submitted_at), MONTH(submitted_at) ORDER BY ticket_year, ticket_month;
SELECT priority, AVG(resolution_time_hours) AS avg_resolution FROM support_tickets GROUP BY priority ORDER BY avg_resolution DESC;
SELECT priority, AVG(satisfaction_score) AS avg_satisfaction FROM support_tickets GROUP BY priority ORDER BY avg_satisfaction;

### 2.5 Sections 8–12: Relational Joins & Combined Business Analysis

In [ ]:
%%sql
-- ============================================================
-- SECTION 8: ACCOUNT + SUBSCRIPTION ANALYSIS (JOINS)
-- ============================================================
SELECT a.account_id, a.account_name, s.subscription_id, s.plan_tier, s.mrr_amount, s.arr_amount
FROM accounts a JOIN subscriptions s ON a.account_id = s.account_id
LIMIT 20;

-- Subscriptions and revenue per account
SELECT a.account_id, a.account_name, COUNT(s.subscription_id) AS num_subscriptions
FROM accounts a JOIN subscriptions s ON a.account_id = s.account_id
GROUP BY a.account_id, a.account_name
ORDER BY num_subscriptions DESC;

SELECT a.account_id, a.account_name, SUM(s.mrr_amount) AS total_mrr, SUM(s.arr_amount) AS total_arr
FROM accounts a JOIN subscriptions s ON a.account_id = s.account_id
GROUP BY a.account_id, a.account_name
ORDER BY total_mrr DESC;

SELECT a.account_id, a.account_name, s.plan_tier, s.churn_flag
FROM accounts a JOIN subscriptions s ON a.account_id = s.account_id
WHERE s.churn_flag = 1;

-- ============================================================
-- SECTION 9: ACCOUNT + CHURN ANALYSIS (JOINS)
-- ============================================================
SELECT a.industry, COUNT(*) AS num_churns
FROM accounts a JOIN churn_events c ON a.account_id = c.account_id
GROUP BY a.industry ORDER BY num_churns DESC;

SELECT a.country, COUNT(*) AS num_churns
FROM accounts a JOIN churn_events c ON a.account_id = c.account_id
GROUP BY a.country ORDER BY num_churns DESC;

SELECT a.plan_tier, COUNT(*) AS num_churns
FROM accounts a JOIN churn_events c ON a.account_id = c.account_id
GROUP BY a.plan_tier ORDER BY num_churns DESC;

SELECT a.referral_source, COUNT(*) AS num_churns
FROM accounts a JOIN churn_events c ON a.account_id = c.account_id
GROUP BY a.referral_source ORDER BY num_churns DESC;

SELECT a.account_id, a.account_name, c.reason_code, c.churn_date
FROM accounts a JOIN churn_events c ON a.account_id = c.account_id;

SELECT a.account_id, a.account_name, c.refund_amount_usd
FROM accounts a JOIN churn_events c ON a.account_id = c.account_id
ORDER BY c.refund_amount_usd DESC;

-- ============================================================
-- SECTION 10: ACCOUNT + SUPPORT ANALYSIS (JOINS)
-- ============================================================
SELECT a.account_id, a.account_name, COUNT(t.ticket_id) AS num_tickets
FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id
GROUP BY a.account_id, a.account_name
ORDER BY num_tickets DESC;

SELECT a.account_id, a.account_name, AVG(t.satisfaction_score) AS avg_satisfaction
FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id
GROUP BY a.account_id, a.account_name
ORDER BY avg_satisfaction ASC;

SELECT a.account_id, a.account_name, AVG(t.resolution_time_hours) AS avg_resolution
FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id
GROUP BY a.account_id, a.account_name
ORDER BY avg_resolution DESC;

SELECT a.account_id, a.account_name, SUM(t.escalation_flag) AS num_escalations
FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id
GROUP BY a.account_id, a.account_name
ORDER BY num_escalations DESC;

SELECT a.industry, COUNT(t.ticket_id) AS num_tickets
FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id
GROUP BY a.industry ORDER BY num_tickets DESC;

SELECT a.plan_tier, COUNT(t.ticket_id) AS num_tickets
FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id
GROUP BY a.plan_tier ORDER BY num_tickets DESC;

-- ============================================================
-- SECTION 11: SUBSCRIPTION + FEATURE USAGE (JOINS)
-- ============================================================
SELECT s.subscription_id, s.plan_tier, f.feature_name, SUM(f.usage_count) AS total_usage
FROM subscriptions s JOIN feature_usage f ON s.subscription_id = f.subscription_id
GROUP BY s.subscription_id, s.plan_tier, f.feature_name
ORDER BY total_usage DESC
LIMIT 20;

SELECT s.plan_tier, f.feature_name, SUM(f.usage_count) AS total_usage
FROM subscriptions s JOIN feature_usage f ON s.subscription_id = f.subscription_id
GROUP BY s.plan_tier, f.feature_name
ORDER BY s.plan_tier, total_usage DESC;

SELECT s.plan_tier, AVG(f.usage_count) AS avg_usage
FROM subscriptions s JOIN feature_usage f ON s.subscription_id = f.subscription_id
GROUP BY s.plan_tier;

SELECT s.plan_tier, SUM(f.error_count) AS total_errors
FROM subscriptions s JOIN feature_usage f ON s.subscription_id = f.subscription_id
GROUP BY s.plan_tier ORDER BY total_errors DESC;

SELECT s.plan_tier, SUM(f.is_beta_feature) AS beta_usage_count
FROM subscriptions s JOIN feature_usage f ON s.subscription_id = f.subscription_id
GROUP BY s.plan_tier ORDER BY beta_usage_count DESC;

-- ============================================================
-- SECTION 12: COMBINED BUSINESS ANALYSIS
-- ============================================================
SELECT s.plan_tier, AVG(f.usage_count) AS avg_usage, AVG(f.error_count) AS avg_errors
FROM subscriptions s JOIN feature_usage f ON s.subscription_id = f.subscription_id
GROUP BY s.plan_tier;

SELECT s.churn_flag, AVG(f.usage_count) AS avg_usage
FROM subscriptions s JOIN feature_usage f ON s.subscription_id = f.subscription_id
GROUP BY s.churn_flag;

SELECT plan_tier, SUM(mrr_amount) AS total_mrr FROM subscriptions GROUP BY plan_tier ORDER BY total_mrr DESC;
SELECT a.plan_tier, COUNT(*) AS num_churns FROM accounts a JOIN churn_events c ON a.account_id = c.account_id GROUP BY a.plan_tier ORDER BY num_churns DESC;
SELECT industry, COUNT(*) AS num_accounts FROM accounts GROUP BY industry ORDER BY num_accounts DESC;
SELECT a.industry, COUNT(*) AS num_churns FROM accounts a JOIN churn_events c ON a.account_id = c.account_id GROUP BY a.industry ORDER BY num_churns DESC;
SELECT a.churn_flag, COUNT(t.ticket_id) AS num_tickets, AVG(t.satisfaction_score) AS avg_satisfaction FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id GROUP BY a.churn_flag;
SELECT a.plan_tier, AVG(t.satisfaction_score) AS avg_satisfaction FROM accounts a JOIN support_tickets t ON a.account_id = t.account_id GROUP BY a.plan_tier;
SELECT reason_code, SUM(preceding_upgrade_flag) AS after_upgrade, SUM(preceding_downgrade_flag) AS after_downgrade FROM churn_events GROUP BY reason_code;
SELECT feature_name, SUM(usage_count) AS total_usage FROM feature_usage GROUP BY feature_name ORDER BY total_usage DESC;
SELECT feature_name, SUM(error_count) AS total_errors FROM feature_usage GROUP BY feature_name ORDER BY total_errors DESC;

---
# PART 3: Data Cleaning, Validation & Export
*(Source: `saas_project.sql`)*

Creation of clean surrogate/filtered tables, row-count checks, orphan relation validation, and exporting cleaned datasets to CSV.

### 3.1 Create Clean Tables (Filtering Anomalies, Null Keys & Range Violations)

In [ ]:
%%sql
-- ============================================================
-- 1. Create clean tables
-- ============================================================

-- Accounts
CREATE TABLE accounts_clean AS SELECT DISTINCT *
FROM accounts WHERE account_id IS NOT NULL;

-- Subscriptions
CREATE TABLE subscriptions_clean AS SELECT DISTINCT *
FROM subscriptions WHERE subscription_id IS NOT NULL
  AND account_id IS NOT NULL
  AND (seats > 0 OR seats IS NULL)
  AND (mrr_amount >= 0 OR mrr_amount IS NULL)
  AND (arr_amount >= 0 OR arr_amount IS NULL)
  AND (end_date >= start_date OR end_date IS NULL);

-- Churn Events
CREATE TABLE churn_events_clean AS SELECT DISTINCT *
FROM churn_events WHERE churn_event_id IS NOT NULL
  AND account_id IS NOT NULL
  AND (refund_amount_usd >= 0 OR refund_amount_usd IS NULL);

-- Feature Usage
CREATE TABLE feature_usage_clean AS SELECT DISTINCT *
FROM feature_usage WHERE usage_id IS NOT NULL
  AND subscription_id IS NOT NULL
  AND (usage_count >= 0 OR usage_count IS NULL)
  AND (usage_duration_secs >= 0 OR usage_duration_secs IS NULL)
  AND (error_count >= 0 OR error_count IS NULL);

-- Support Tickets
CREATE TABLE support_tickets_clean AS SELECT DISTINCT *
FROM support_tickets WHERE ticket_id IS NOT NULL
  AND account_id IS NOT NULL
  AND (resolution_time_hours >= 0 OR resolution_time_hours IS NULL)
  AND (first_response_time_minutes >= 0 OR first_response_time_minutes IS NULL)
  AND (satisfaction_score BETWEEN 0 AND 5 OR satisfaction_score IS NULL)
  AND (closed_at >= submitted_at OR closed_at IS NULL);

### 3.2 Verify Clean Tables & Row Counts

In [ ]:
%%sql
-- ============================================================
-- 2. Check whether the clean tables exist
-- ============================================================
SHOW TABLES;

-- ============================================================
-- 3. Check row counts
-- ============================================================
SELECT 'accounts_clean' AS table_name, COUNT(*) AS total_rows FROM accounts_clean
UNION ALL
SELECT 'subscriptions_clean', COUNT(*) FROM subscriptions_clean
UNION ALL
SELECT 'churn_events_clean', COUNT(*) FROM churn_events_clean
UNION ALL
SELECT 'feature_usage_clean', COUNT(*) FROM feature_usage_clean
UNION ALL
SELECT 'support_tickets_clean', COUNT(*) FROM support_tickets_clean;

### 3.3 Post-Clean Duplicate Checks

In [ ]:
%%sql
-- ============================================================
-- 4. Check duplicates again
-- ============================================================
-- Accounts
SELECT account_id, COUNT(*) AS cnt FROM accounts_clean
GROUP BY account_id HAVING COUNT(*) > 1;

-- Subscriptions
SELECT subscription_id, COUNT(*) AS cnt
FROM subscriptions_clean
GROUP BY subscription_id
HAVING COUNT(*) > 1;

-- Churn
SELECT churn_event_id, COUNT(*) AS cnt
FROM churn_events_clean
GROUP BY churn_event_id
HAVING COUNT(*) > 1;

-- Feature usage
SELECT usage_id, COUNT(*) AS cnt
FROM feature_usage_clean
GROUP BY usage_id
HAVING COUNT(*) > 1;

-- Tickets
SELECT ticket_id, COUNT(*) AS cnt
FROM support_tickets_clean
GROUP BY ticket_id
HAVING COUNT(*) > 1;

### 3.4 Referential Integrity (Orphan Record Checks)

In [ ]:
%%sql
-- ============================================================
-- 5. Check relationships (Orphans)
-- ============================================================
-- Check subscriptions:
SELECT COUNT(*) AS orphan_subscriptions
FROM subscriptions_clean s
LEFT JOIN accounts_clean a
    ON s.account_id = a.account_id
WHERE a.account_id IS NULL;

-- Check churn:
SELECT COUNT(*) AS orphan_churn_events
FROM churn_events_clean c
LEFT JOIN accounts_clean a
    ON c.account_id = a.account_id
WHERE a.account_id IS NULL;

-- Check feature usage:
SELECT COUNT(*) AS orphan_usage
FROM feature_usage_clean f
LEFT JOIN subscriptions_clean s
    ON f.subscription_id = s.subscription_id
WHERE s.subscription_id IS NULL;

-- Check tickets:
SELECT COUNT(*) AS orphan_tickets
FROM support_tickets_clean t
LEFT JOIN accounts_clean a
    ON t.account_id = a.account_id
WHERE a.account_id IS NULL;

### 3.5 Export Cleaned Tables to CSV (`INTO OUTFILE`)

In [ ]:
%%sql
-- ============================================================
-- 6. Now save the clean tables as CSV
-- ============================================================
-- Accounts
SELECT * FROM accounts_clean
INTO OUTFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/accounts_clean.csv'
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n';

-- Subscriptions
SELECT * FROM subscriptions_clean
INTO OUTFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/subscriptions_clean.csv'
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n';

-- Churn Events
SELECT * FROM churn_events_clean
INTO OUTFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/churn_events_clean.csv'
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n';

-- Feature Usage
SELECT * FROM feature_usage_clean
INTO OUTFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/feature_usage_clean.csv'
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n';

-- Support Tickets
SELECT *
FROM support_tickets_clean
INTO OUTFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/support_tickets_clean.csv'
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n';